# CVD multi-condition transfer analysis

## tl;dr
Conditions [1, 2, 4, 5] identify the response model; condition 3 is held out without refitting. Numerical prediction winner: cvd:aib_qss:AIB:full:bulk_as_surface:A=idn_2,I=n2,B=adn_2. Adopted model: none. Response structure: surface_qss. Decision: review; role support: unresolved. Outer condition folds assess the selection procedure; each fold fits its own model.


## Context & Methods

Quasi-steady site-balance families use absolute species concentrations normalized by identification-data references. Training-condition CV selects the equation family, exact reduction, and anonymous-species role assignment. Term removal and alternative assignments are compared on the same conditions; crossing losses leave role support unresolved. Numerical loss ties prefer fewer effects and parameters. Angular/radial blocked CV and design rank are diagnostics.

### Key Assumptions
- Response coefficients transfer across conditions; there are no measured-rate corrections for an unseen condition.
- The test condition is not used for fitting or model selection.
- Coefficients are effective transfer responses, not elementary kinetics.
- The selected concentration mode is recorded in analysis_summary.json; absolute wall flux is not calculated.


## Data


In [1]:
import csv, json
from pathlib import Path
output = Path('.')
with (output / 'analysis_summary.json').open(encoding='utf-8') as handle:
    summary = json.load(handle)
with (output / 'condition_quality.csv').open(encoding='utf-8') as handle:
    quality = list(csv.DictReader(handle))
print('train/test:', summary['primary_split']['train_cases'], '->', summary['primary_split']['test_case'])
print('condition rows:', [(row['condition'], row['rows'], row['rate_unique_count']) for row in quality])


train/test: 1+2+4+5 -> 3
condition rows: [('1', '49', '49'), ('2', '49', '49'), ('3', '49', '49'), ('4', '49', '49'), ('5', '49', '49')]


## Results


In [2]:
primary = summary['primary_split']
print('numerical prediction winner:', primary['selected_model'])
print('adopted model:', summary['validity'].get('adopted_model') or 'none')
print('common order:', primary['common_total_order'])
print('test RMSE [nm/s]:', primary['test_rmse_nm_s'])
print('test relative RMSE:', primary['test_relative_rmse_vs_test_mean'])
print('test spatial R2:', primary['test_centered_spatial_r2'])
print('species-role assessment:', summary['validity']['species_role_assessment'])
print('model-structure envelope / test mean:', summary['model_structure_uncertainty']['mean_envelope_width_relative_to_test_mean'])
print('equation families:')
for row in summary.get('equation_family_assessments', []):
    print(row['equation_family'], row['applicability_status'], row['condition_cv_rmse_nm_s'], row['outer_selection_frequency'])
print('reaction mechanisms:')
for row in summary.get('reaction_mechanism_assessments', []):
    print(row['mechanism_id'], row['evaluation_status'], row['steady_representation'])


numerical prediction winner: cvd:aib_qss:AIB:full:bulk_as_surface:A=idn_2,I=n2,B=adn_2
adopted model: none
common order: None
test RMSE [nm/s]: 0.0010486307401464674
test relative RMSE: 0.007286473490638807
test spatial R2: -0.014799642556980741
species-role assessment: unresolved
model-structure envelope / test mean: 0.002983027233014149
equation families:
aib_qss production 0.0008940841559704123 0.6
parallel_a_ab_qss production 0.0009033381894203596 0.0
langmuir_hinshelwood_qss exploratory 0.0009259607308688309 0.4
reaction mechanisms:
aib_qss fitted cvd:aib_qss:AIB:full:bulk_as_surface:A=idn_2,I=n2,B=adn_2
parallel_a_ab_qss fitted cvd:parallel_a_ab_qss:AB:full:bulk_as_surface:A=adn_2,B=idn_2
langmuir_hinshelwood_qss fitted cvd:langmuir_hinshelwood_qss:AB:full:bulk_as_surface:A=adn_2,B=idn_2
mars_van_krevelen steady_observable_equivalent cvd:aib_qss:AB:no_desorption:bulk_as_surface:A=adn_2,B=n2


### Figures

![Condition transfer](plots/condition_mean_transfer.png)

![Held-out fit](plots/test_measured_vs_predicted.png)

![Held-out maps](plots/test_spatial_maps.png)


## Takeaways

- Fixed-model holdout: improves_baseline; spatial shape: not_supported.
- Outer selection procedure: improves_baseline. Application criteria: not_specified.
- Raw species are candidate inputs. An unresolved steady AB response does not determine its A/B direction.
- Decision evidence: prediction does not explain within-condition spatial variation; term removal has not shown consistent additional predictive benefit across conditions; alternative raw-species assignments are not distinguished across conditions; assigned species lack independent between-condition excitation; effective roles change across training-condition selections; selected equation family, reduction, or role structure changes across outer condition splits; application scope/error tolerance: not_specified.
